In [0]:
# Read datasets

customers = spark.read.option("header", "true").csv("/Volumes/workspace/default/olist/olist_customers_dataset.csv")

orders = spark.read.option("header", "true").csv("/Volumes/workspace/default/olist/olist_orders_dataset.csv")

order_items = spark.read.option("header", "true").csv("/Volumes/workspace/default/olist/olist_order_items_dataset.csv")

payments = spark.read.option("header", "true").csv("/Volumes/workspace/default/olist/olist_order_payments_dataset.csv")

products = spark.read.option("header", "true").csv("/Volumes/workspace/default/olist/olist_products_dataset.csv")

sellers = spark.read.option("header", "true").csv("/Volumes/workspace/default/olist/olist_sellers_dataset.csv")

reviews = spark.read.option("header", "true").csv("/Volumes/workspace/default/olist/olist_order_reviews_dataset.csv")

category = spark.read.option("header", "true").csv("/Volumes/workspace/default/olist/product_category_name_translation.csv")

In [0]:
from pyspark.sql.functions import col
customers=customers.dropDuplicates()
orders=orders.dropDuplicates()
order_items=order_items.dropDuplicates()

#Remove Null keys
customers=customers.dropna(subset=["customer_id"])
orders=orders.dropna(subset=["order_id","customer_id"])
order_items=order_items.dropna(subset=["order_id","product_id"])

In [0]:
df=orders.join(customers,"customer_id").join(order_items,"order_id").join(products,"product_id")

%md
**Step 2: Analytical Tasks (Detailed)**

**Task 1: Top 3 Customers per City Calculate total spend per customer, then use window function to rank customers within each city. Output: city, customer_id, total_spend**

In [0]:

from pyspark.sql.functions import sum,rank
from pyspark.sql.window import Window
customer_spend=df.groupBy("customer_city","customer_id").agg(sum("price").alias("total_spend"))

window_city=Window.partitionBy("customer_city").orderBy(col("total_spend").desc())
top_customers=customer_spend.withColumn("rank",rank().over(window_city)).filter(col("rank")<=3)

top_customers.show()


+-------------------+--------------------+-----------+----+
|      customer_city|         customer_id|total_spend|rank|
+-------------------+--------------------+-----------+----+
|abadia dos dourados|9e01f714a2b3b8962...|      199.0|   1|
|abadia dos dourados|a23e3f9a2b656b23b...|      120.0|   2|
|abadia dos dourados|f11eb8f0b8b87510a...|       39.9|   3|
|          abadiania|576d71ddb21b21763...|     949.99|   1|
|             abaete|d47c8bb51df6f716e...|      449.0|   1|
|             abaete|ff0d62f8be4c098e6...|      225.9|   2|
|             abaete|5371894984937a27c...|      208.9|   3|
|         abaetetuba|c7eb06383ae604616...|     1500.0|   1|
|         abaetetuba|367fd22f1e994de87...|      797.6|   2|
|         abaetetuba|7727e2cc9ec428ad3...|     580.27|   3|
|            abaiara|f8508e9ec506a046a...|      169.0|   1|
|            abaiara|9723d86b12bdec025...|       93.9|   2|
|             abaira|11888d3334232a1eb...|      145.0|   1|
|             abaira|43b0018187e283119..

**Task 2: Running Total of Sales Calculate daily sales and then cumulative (running) total using window
function. Output: date, daily_sales, running_total**

**Task 2: Running Total of Sales**

In [0]:
from pyspark.sql.functions import to_date

df = df.withColumn("order_date", to_date(col("order_purchase_timestamp")))

daily_sales = df.groupBy("order_date") \
    .agg(sum("price").alias("daily_sales"))

window_date = Window.orderBy("order_date")

running_total = daily_sales.withColumn("running_total", sum("daily_sales").over(window_date))

running_total.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+----------+------------------+------------------+
|order_date|       daily_sales|     running_total|
+----------+------------------+------------------+
|2016-09-04|             72.89|             72.89|
|2016-09-05|              59.5|            132.39|
|2016-09-15|            134.97|            267.36|
|2016-10-02|             100.0|            367.36|
|2016-10-03| 463.4800000000001| 830.8400000000001|
|2016-10-04| 9940.960000000001|10771.800000000001|
|2016-10-05|           8343.25|19115.050000000003|
|2016-10-06| 7960.509999999999|          27075.56|
|2016-10-07| 7228.049999999999|          34303.61|
|2016-10-08| 8441.849999999999|          42745.46|
|2016-10-09|3336.9899999999993|          46082.45|
|2016-10-10|3692.5699999999993|          49775.02|
|2016-12-23|              10.9|          49785.92|
|2017-01-05|             396.9|          50182.82|
|2017-01-06|            916.38|           51099.2|
|2017-01-07|            1351.9|           52451.1|
|2017-01-08|            709.58|

**Task 3: Top Products per Category Aggregate sales per product, join with category, then rank using
DENSE_RANK. Output: category, product_id, total_sales, rank**

In [0]:
from pyspark.sql.functions import dense_rank

product_sales = df.groupBy("product_id", "product_category_name") \
    .agg(sum("price").alias("total_sales"))

# Join with category translation
product_sales = product_sales.join(category, "product_category_name")

window_category = Window.partitionBy("product_category_name_english") \
    .orderBy(col("total_sales").desc())

top_products = product_sales.withColumn("rank", dense_rank().over(window_category))

top_products.show()

+---------------------+--------------------+-----------+-----------------------------+----+
|product_category_name|          product_id|total_sales|product_category_name_english|rank|
+---------------------+--------------------+-----------+-----------------------------+----+
| agro_industria_e_...|11250b0d4b709fee9...|     9111.0|         agro_industry_and...|   1|
| agro_industria_e_...|423a6644f0aa529e8...|     8043.0|         agro_industry_and...|   2|
| agro_industria_e_...|672e757f331900b9d...|     6885.0|         agro_industry_and...|   3|
| agro_industria_e_...|c183fd5d2abf05873...|     5934.6|         agro_industry_and...|   4|
| agro_industria_e_...|2b69866f22de8dad6...|     2990.0|         agro_industry_and...|   5|
| agro_industria_e_...|c89226b8a795ae3d6...|     2821.5|         agro_industry_and...|   6|
| agro_industria_e_...|5fb0955cb683eb6f6...|     2720.0|         agro_industry_and...|   7|
| agro_industria_e_...|b7a60a397d4efd05c...|     2399.0|         agro_industry_a

**Task 4: Customer Lifetime Value**

In [0]:
clv = df.groupBy("customer_id") \
    .agg(sum("price").alias("total_spend"))

clv.show()

+--------------------+-----------+
|         customer_id|total_spend|
+--------------------+-----------+
|0a924ae136218f3bb...|      449.0|
|7186038cf7f3f1ef5...|    1039.97|
|e6cd6ca84cc5b5890...|     199.99|
|5cbf1c7456d9789a9...|      13.99|
|91b9c73efce9cbe8c...|      539.0|
|1db709d59de22eef7...|      229.9|
|8b036273a1a512da2...|      188.9|
|a0edccf84fb09993f...|     155.75|
|ff4ead15d934b5b58...|      59.99|
|5f46e687a5c82cb1e...|      19.99|
|77305d99b811c5927...|       89.9|
|7a72e1fbb23c6558e...|      56.97|
|fa171236769262c2d...|       21.5|
|b82342c1221c839ac...|       29.9|
|2656a50d0f9e0263c...|       75.0|
|3d5cd39f5aa33e007...|      117.8|
|e7a055b4b4b6a31ca...|     229.99|
|8d2feb200cbe341f3...|       35.9|
|81a742c53ac122ce2...|     129.99|
|300ce7fc3fb7ffda8...|       89.9|
+--------------------+-----------+
only showing top 20 rows


**Task 5: Customer Segmentation**

In [0]:
from pyspark.sql.functions import when

segmentation = clv.withColumn(
    "segment",
    when(col("total_spend") > 10000, "Gold")
    .when((col("total_spend") >= 5000) & (col("total_spend") <= 10000), "Silver")
    .otherwise("Bronze")
)

segmentation.show()

+--------------------+-----------+-------+
|         customer_id|total_spend|segment|
+--------------------+-----------+-------+
|0a924ae136218f3bb...|      449.0| Bronze|
|7186038cf7f3f1ef5...|    1039.97| Bronze|
|e6cd6ca84cc5b5890...|     199.99| Bronze|
|5cbf1c7456d9789a9...|      13.99| Bronze|
|91b9c73efce9cbe8c...|      539.0| Bronze|
|1db709d59de22eef7...|      229.9| Bronze|
|8b036273a1a512da2...|      188.9| Bronze|
|a0edccf84fb09993f...|     155.75| Bronze|
|ff4ead15d934b5b58...|      59.99| Bronze|
|5f46e687a5c82cb1e...|      19.99| Bronze|
|77305d99b811c5927...|       89.9| Bronze|
|7a72e1fbb23c6558e...|      56.97| Bronze|
|fa171236769262c2d...|       21.5| Bronze|
|b82342c1221c839ac...|       29.9| Bronze|
|2656a50d0f9e0263c...|       75.0| Bronze|
|3d5cd39f5aa33e007...|      117.8| Bronze|
|e7a055b4b4b6a31ca...|     229.99| Bronze|
|8d2feb200cbe341f3...|       35.9| Bronze|
|81a742c53ac122ce2...|     129.99| Bronze|
|300ce7fc3fb7ffda8...|       89.9| Bronze|
+----------

**Task 6: Final Reporting Table**

In [0]:
from pyspark.sql.functions import count

total_orders = orders.groupBy("customer_id") \
    .agg(count("order_id").alias("total_orders"))

final_df = segmentation.join(customers, "customer_id") \
    .join(total_orders, "customer_id") \
    .select(
        "customer_id",
        "customer_city",
        "total_spend",
        "segment",
        "total_orders"
    )

final_df.show()

+--------------------+--------------------+-----------+-------+------------+
|         customer_id|       customer_city|total_spend|segment|total_orders|
+--------------------+--------------------+-----------+-------+------------+
|0f4680c4ae80eb5eb...|sao bernardo do c...|      149.9| Bronze|           1|
|90b1c0370ada0b358...|      rio de janeiro|       69.9| Bronze|           1|
|ac90434fffc630332...|      rio de janeiro|      118.9| Bronze|           1|
|cb0e9523a4cddb065...|          mogi-guacu|       75.9| Bronze|           1|
|4adeaee8338a42ef0...|           sao paulo|     379.94| Bronze|           1|
|684fa6da5134b9e4d...|             aracaju|      45.98| Bronze|           1|
|a31d81182a35974e4...|         barra mansa|      64.89| Bronze|           1|
|37d31448db0f51f0e...|           sao paulo|       54.9| Bronze|           1|
|5a1c1e9fe3aa6fa5a...|           sao paulo|       78.8| Bronze|           1|
|b8cc30d4bb91f7718...|           sao paulo|       59.9| Bronze|           1|